# Algorithm Library

A catalog of ready-to-use `DynamicalSystem` building blocks.
Each entry shows the equations, the `f`/`h` implementation, and a minimal usage example.

## How to contribute

Every algorithm follows the same four-part template:

1. **Markdown cell** — name, one-line description, equations for `f` and `h`.
2. **Dependencies cell** (if needed) — imports or helper functions the block relies on.
3. **Implementation cell** — define `f`, `h`, and instantiate the `DynamicalSystem`.
4. **Usage cell** — a minimal `block.step(...)` call showing inputs and outputs.

To add a new algorithm, copy the four cells from the nearest similar entry and fill in your own equations and code.

---

**Contents**

- [Estimators](#estimators)
  - [Kalman Filter (KF)](#kalman-filter-kf)
  - [Extended Kalman Filter (EKF)](#extended-kalman-filter-ekf)
  - [Unscented Kalman Filter (UKF)](#unscented-kalman-filter-ukf)
- [Controllers](#controllers)
  - [PID Controller](#pid-controller)
  - [LQR Controller](#lqr-controller)

In [ ]:
import numpy as np
from dynamicalnodes import DynamicalSystem

---

## Estimators

Estimators are `DynamicalSystem` blocks whose state is the pair $(\hat{x}_k, P_k)$:
the current state estimate and its covariance matrix.
`h` always returns $\hat{x}_k$ so the estimate can be passed directly to a controller.

### Kalman Filter (KF)

Optimal estimator for **linear, Gaussian** systems.

**State:** $z_k = (\hat{x}_k,\, P_k)$

**Predict:**
$$\hat{x}_{k|k-1} = F_k\hat{x}_{k-1} + B_k u_k, \qquad P_{k|k-1} = F_k P_{k-1} F_k^\top + Q_k$$

**Update:**
$$K_k = P_{k|k-1} H_k^\top S_k^{-1}, \quad
\hat{x}_k = \hat{x}_{k|k-1} + K_k(y_k - H_k\hat{x}_{k|k-1}), \quad
P_k = (I - K_k H_k)\,P_{k|k-1}$$

where $S_k = H_k P_{k|k-1} H_k^\top + R_k$.

**Inputs to `step`:** `zk`, `uk`, `yk`, `Fk`, `Bk`, `Hk`, `Qk`, `Rk`  
**Dependencies:** `numpy`

In [ ]:
def kf_f(zk, uk, yk, Fk, Bk, Hk, Qk, Rk):
    x, P = zk
    x_pred = Fk @ x + Bk * uk
    P_pred = Fk @ P @ Fk.T + Qk
    S = Hk @ P_pred @ Hk.T + Rk
    K = P_pred @ Hk.T @ np.linalg.inv(S)
    yk = np.atleast_1d(yk)
    x_upd = x_pred + K @ (yk - Hk @ x_pred)
    P_upd = (np.eye(len(x)) - K @ Hk) @ P_pred
    return (x_upd, P_upd)

def kf_h(zk, **kwargs):
    return zk[0]

kf = DynamicalSystem(f=kf_f, h=kf_h)

In [ ]:
# Usage — 1-D constant-velocity model, velocity observed
dt = 0.1
Fk = np.array([[1., dt], [0., 1.]])
Bk = np.array([0., dt])
Hk = np.array([[0., 1.]])
Qk = np.diag([0.01, 0.1])
Rk = np.array([[1.0]])

zk = (np.zeros(2), np.eye(2))
zk_next, x_est = kf.step(zk=zk, uk=1.0, yk=np.array([0.12]),
                          Fk=Fk, Bk=Bk, Hk=Hk, Qk=Qk, Rk=Rk)
print("estimate:", x_est)      # [position_est, velocity_est]
print("covariance:\n", zk_next[1])

### Extended Kalman Filter (EKF)

Extends the KF to **nonlinear** systems by linearising around the current estimate
using user-supplied Jacobians $F^J_k = \partial f / \partial x$ and $H^J_k = \partial h / \partial x$.

**State:** $z_k = (\hat{x}_k,\, P_k)$

**Predict:**
$$\hat{x}_{k|k-1} = f(\hat{x}_{k-1}, u_k), \qquad
P_{k|k-1} = F^J_k P_{k-1} {F^J_k}^\top + Q_k$$

**Update:** same as KF but with $F^J_k$, $H^J_k$ in place of $F_k$, $H_k$.

**Inputs to `step`:** `zk`, `uk`, `yk`, `f_nl`, `h_nl`, `Fjk`, `Hjk`, `Qk`, `Rk`  
**Dependencies:** `numpy`

In [ ]:
def ekf_f(zk, uk, yk, f_nl, h_nl, Fjk, Hjk, Qk, Rk):
    x, P = zk
    x_pred = f_nl(x, uk)
    P_pred = Fjk @ P @ Fjk.T + Qk
    S = Hjk @ P_pred @ Hjk.T + Rk
    K = P_pred @ Hjk.T @ np.linalg.inv(S)
    yk = np.atleast_1d(yk)
    x_upd = x_pred + K @ (yk - h_nl(x_pred))
    P_upd = (np.eye(len(x)) - K @ Hjk) @ P_pred
    return (x_upd, P_upd)

def ekf_h(zk, **kwargs):
    return zk[0]

ekf = DynamicalSystem(f=ekf_f, h=ekf_h)

In [ ]:
# Usage — pendulum: x = [theta, theta_dot], observe theta
g, L, dt = 9.81, 1.0, 0.05

def pend_f(x, u):
    th, w = x
    return np.array([th + dt * w,
                     w  + dt * (-g / L * np.sin(th) + u)])

def pend_h(x):
    return np.array([x[0]])

def pend_Fj(x):
    th, w = x
    return np.array([[1.,              dt],
                     [-dt * g / L * np.cos(th), 1.]])

Hjk = np.array([[1., 0.]])
Qk  = np.diag([1e-4, 1e-3])
Rk  = np.array([[0.05]])

zk  = (np.array([0.1, 0.0]), np.eye(2))
x0  = zk[0]
zk_next, x_est = ekf.step(zk=zk, uk=0.0,
                           yk=np.array([0.09]),
                           f_nl=pend_f, h_nl=pend_h,
                           Fjk=pend_Fj(x0), Hjk=Hjk,
                           Qk=Qk, Rk=Rk)
print("estimate:", x_est)   # [theta_est, theta_dot_est]

### Unscented Kalman Filter (UKF)

Handles **nonlinear** systems without Jacobians by propagating a set of deterministically
chosen **sigma points** through the true nonlinear functions.

**State:** $z_k = (\hat{x}_k,\, P_k)$

**Sigma points** ($n$ = state dimension, $\lambda = \alpha^2(n+\kappa)-n$):
$$\mathcal{X}^{(0)} = \hat{x}_{k-1}, \quad
\mathcal{X}^{(i)} = \hat{x}_{k-1} \pm \bigl(\sqrt{(n+\lambda)P_{k-1}}\bigr)_i, \quad i=1,\ldots,n$$

**Predict / Update:** weighted mean and covariance of propagated sigma points,
then standard Kalman gain and correction.

**Inputs to `step`:** `zk`, `uk`, `yk`, `f_nl`, `h_nl`, `Qk`, `Rk`  
**Optional:** `alpha` (default `1e-3`), `beta` (default `2.0`), `kappa` (default `0.0`)  
**Dependencies:** `numpy`

In [ ]:
def ukf_f(zk, uk, yk, f_nl, h_nl, Qk, Rk,
          alpha=1e-3, beta=2.0, kappa=0.0):
    x, P = zk
    n   = len(x)
    lam = alpha**2 * (n + kappa) - n

    sqrtP  = np.linalg.cholesky((n + lam) * P)
    sigmas = np.column_stack(
        [x]
        + [x + sqrtP[:, i] for i in range(n)]
        + [x - sqrtP[:, i] for i in range(n)]
    )
    Wm    = np.full(2 * n + 1, 1.0 / (2 * (n + lam)))
    Wm[0] = lam / (n + lam)
    Wc    = Wm.copy()
    Wc[0] += 1 - alpha**2 + beta

    # Predict
    sf = np.column_stack([f_nl(sigmas[:, i], uk) for i in range(2 * n + 1)])
    xp = sf @ Wm
    Pp = sum(Wc[i] * np.outer(sf[:, i] - xp, sf[:, i] - xp)
             for i in range(2 * n + 1)) + Qk

    # Update
    yk = np.atleast_1d(yk)
    sh = np.column_stack([np.atleast_1d(h_nl(sf[:, i])) for i in range(2 * n + 1)])
    yp = sh @ Wm
    S  = sum(Wc[i] * np.outer(sh[:, i] - yp, sh[:, i] - yp)
             for i in range(2 * n + 1)) + Rk
    Pxy = sum(Wc[i] * np.outer(sf[:, i] - xp, sh[:, i] - yp)
              for i in range(2 * n + 1))
    K   = Pxy @ np.linalg.inv(S)
    return (xp + K @ (yk - yp), Pp - K @ S @ K.T)

def ukf_h(zk, **kwargs):
    return zk[0]

ukf = DynamicalSystem(f=ukf_f, h=ukf_h)

In [ ]:
# Usage — same pendulum as EKF example, no Jacobian required
Qk = np.diag([1e-4, 1e-3])
Rk = np.array([[0.05]])

zk = (np.array([0.1, 0.0]), np.eye(2))
zk_next, x_est = ukf.step(zk=zk, uk=0.0,
                           yk=np.array([0.09]),
                           f_nl=pend_f, h_nl=pend_h,
                           Qk=Qk, Rk=Rk)
print("estimate:", x_est)   # [theta_est, theta_dot_est]

---

## Controllers

Controllers map a reference $r_k$ and a state estimate $\hat{x}_k$ to a control input $u_k$.
`h` always returns $u_k$ so the output can be fed directly into a plant block.

### PID Controller

Classic **proportional–integral–derivative** controller.

**State:** $c_k = (e_{k-1},\; \textstyle\sum e_k\,\Delta t)$ — previous error and accumulated integral.

$$u_k = K_P\,e_k + K_I\sum e_k\,\Delta t + K_D\,\frac{e_k - e_{k-1}}{\Delta t}, \qquad e_k = r_k - \hat{x}_k$$

**Inputs to `step`:** `ck`, `rk`, `xhatk`, `KP`, `KI`, `KD`, `dt`  
**Dependencies:** `numpy`

In [ ]:
def pid_f(ck, rk, xhatk, KP, KI, KD, dt):
    e_prev, e_int = ck
    ek = rk - xhatk
    return (ek, e_int + ek * dt)

def pid_h(ck, rk, xhatk, KP, KI, KD, dt):
    e_prev, e_int = ck
    ek = rk - xhatk
    return KP * ek + KI * e_int + KD * (ek - e_prev) / dt

pid = DynamicalSystem(f=pid_f, h=pid_h)

In [ ]:
# Usage — single step, reference 10 m/s, current estimate 0 m/s
ck = (0.0, 0.0)   # (e_prev, e_int)
ck_next, uk = pid.step(ck=ck, rk=10.0, xhatk=0.0,
                        KP=2.0, KI=0.5, KD=0.1, dt=0.05)
print("control output u:", uk)
print("updated PID state:", ck_next)

### LQR Controller

**Linear Quadratic Regulator** — optimal state-feedback for linear systems.
The gain matrix $K$ is computed offline via the discrete algebraic Riccati equation (DARE)
and held constant during operation.

**Stateless** — no internal state; `f` is not used.

$$u_k = -K\,(\hat{x}_k - x^{\text{ref}}_k)$$

$$K = (R + B^\top P B)^{-1} B^\top P A, \qquad P = \text{DARE}(A, B, Q, R)$$

**Inputs to `step`:** `xhatk`, `xrefk`, `K`  
**Helper:** `lqr_gain(A, B, Q, R)` — call once before the simulation loop.  
**Dependencies:** `numpy`, `scipy`

In [ ]:
from scipy.linalg import solve_discrete_are

def lqr_gain(A, B, Q, R):
    """Compute discrete-time LQR gain K = (R + B'PB)^{-1} B'PA."""
    P = solve_discrete_are(A, B, Q, R)
    return np.linalg.inv(R + B.T @ P @ B) @ B.T @ P @ A

def lqr_h(xhatk, xrefk, K):
    return -K @ (xhatk - xrefk)

lqr = DynamicalSystem(h=lqr_h)

In [ ]:
# Usage — double-integrator (position + velocity), penalise velocity error
dt = 0.1
A = np.array([[1., dt], [0., 1.]])
B = np.array([[0.], [dt]])
Q = np.diag([0.0, 100.0])   # penalise velocity error
R = np.array([[1e-2]])

K = lqr_gain(A, B, Q, R)
print("LQR gain K:", K)

xhatk = np.array([0.0, 0.0])    # current estimate
xrefk = np.array([0.0, 5.0])    # reference: 5 m/s
uk = lqr.step(xhatk=xhatk, xrefk=xrefk, K=K)
print("control output u:", uk)